In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# -------------------------
# Load Dataset
# -------------------------
df = pd.read_csv("/home/minato/project/main project/Crop-Type-Mapping-using-Multi-Season-Satellite-Data-ML-main/Karnal_Kharif_Training_Data.csv")

# -------------------------
# Remove unwanted columns
# -------------------------
df = df.drop(columns=["system:index", ".geo"])

# -------------------------
# Features & Labels
# -------------------------
X = df.drop("class", axis=1)
y = df["class"]

# -------------------------
# Train-Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# -------------------------
# Random Forest
# -------------------------
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# -------------------------
# Prediction
# -------------------------
pred = model.predict(X_test)

# -------------------------
# Accuracy
# -------------------------
print("\nAccuracy")
print(accuracy_score(y_test, pred))

print("\nClassification Report")
print(classification_report(y_test, pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, pred))

# -------------------------
# Feature Importance
# -------------------------
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\nFeature Importance")
print(importance)

# -------------------------
# Save Model
# -------------------------
joblib.dump(model, "Karnal_RF_Model.pkl")

print("\nModel Saved Successfully")


Accuracy
0.7835616438356164

Classification Report
              precision    recall  f1-score   support

           0       0.78      0.86      0.82       300
           1       0.80      0.66      0.73       130
           2       0.59      0.53      0.56       150
           3       0.95      0.99      0.97       150

    accuracy                           0.78       730
   macro avg       0.78      0.76      0.77       730
weighted avg       0.78      0.78      0.78       730


Confusion Matrix
[[258  10  28   4]
 [ 17  86  27   0]
 [ 56  11  79   4]
 [  1   0   0 149]]

Feature Importance
  Feature  Importance
2      B2    0.142711
8    NDWI    0.132786
0     B11    0.107045
1     B12    0.104816
7    NDVI    0.095453
5      B8    0.089003
9    SAVI    0.088590
4      B4    0.082627
3      B3    0.081770
6    NDBI    0.075199

Model Saved Successfully


In [2]:
import rasterio
import numpy as np
import os

# =====================================================
# Folder containing all exported rasters
# =====================================================
raster_dir = "/home/minato/project/main project/Crop-Type-Mapping-using-Multi-Season-Satellite-Data-ML-main/outputs/rasters"

# =====================================================
# File paths
# =====================================================
bands_file = os.path.join(raster_dir, "Karnal_Kharif_SelectedBands.tif")
ndvi_file  = os.path.join(raster_dir, "Karnal_Kharif_NDVI.tif")
ndwi_file  = os.path.join(raster_dir, "Karnal_Kharif_NDWI.tif")
ndbi_file  = os.path.join(raster_dir, "Karnal_Kharif_NDBI.tif")
savi_file  = os.path.join(raster_dir, "Karnal_Kharif_SAVI.tif")

# =====================================================
# Read Sentinel-2 bands
# =====================================================
with rasterio.open(bands_file) as src:
    bands = src.read()          # (6, rows, cols)
    profile = src.profile
    rows = src.height
    cols = src.width

print("Sentinel Bands Shape :", bands.shape)

# =====================================================
# Read Spectral Indices
# =====================================================
with rasterio.open(ndvi_file) as src:
    ndvi = src.read(1)

with rasterio.open(ndwi_file) as src:
    ndwi = src.read(1)

with rasterio.open(ndbi_file) as src:
    ndbi = src.read(1)

with rasterio.open(savi_file) as src:
    savi = src.read(1)

print("NDVI Shape :", ndvi.shape)
print("NDWI Shape :", ndwi.shape)
print("NDBI Shape :", ndbi.shape)
print("SAVI Shape :", savi.shape)

# =====================================================
# Create 10-band Feature Stack
# Order MUST match training CSV
# B2 B3 B4 B8 B11 B12 NDVI NDWI NDBI SAVI
# =====================================================
feature_stack = np.concatenate([
    bands,
    ndvi[np.newaxis, :, :],
    ndwi[np.newaxis, :, :],
    ndbi[np.newaxis, :, :],
    savi[np.newaxis, :, :]
], axis=0)

print("\nFeature Stack Shape :", feature_stack.shape)
print("Rows :", rows)
print("Cols :", cols)

# =====================================================
# Check
# =====================================================
assert feature_stack.shape[0] == 10, "Error: Feature stack must contain exactly 10 bands."

print("\n✅ Feature stack created successfully.")

Sentinel Bands Shape : (6, 6122, 8118)


/tmp/ipykernel_9156/593244671.py:34: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  ndvi = src.read(1)
/tmp/ipykernel_9156/593244671.py:37: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  ndwi = src.read(1)
/tmp/ipykernel_9156/593244671.py:40: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  ndbi = src.read(1)
/tmp/ipykernel_9156/593244671.py:43: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  savi = src.read(1)


NDVI Shape : (6122, 8118)
NDWI Shape : (6122, 8118)
NDBI Shape : (6122, 8118)
SAVI Shape : (6122, 8118)

Feature Stack Shape : (10, 6122, 8118)
Rows : 6122
Cols : 8118

✅ Feature stack created successfully.


In [ ]:
import numpy as np
import joblib

# =====================================================
# Load trained Random Forest model
# =====================================================
model_path = "Karnal_RF_Model.pkl"

rf = joblib.load(model_path)

print("✅ Random Forest model loaded.")

# =====================================================
# Convert feature stack into ML format
# (bands, rows, cols) -> (pixels, bands)
# =====================================================
X = feature_stack.reshape(10, -1).T

print("Feature Matrix:", X.shape)

# =====================================================
# Remove NaN / Inf values
# =====================================================
X = np.nan_to_num(
    X,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

print("Starting prediction...")

prediction = rf.predict(X)

print("Prediction completed.")

classified = prediction.reshape(rows, cols)

print("Classification Shape:", classified.shape)

✅ Random Forest model loaded.
Feature Matrix: (49698396, 10)
Starting prediction...


/home/minato/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
